# TokenLeak — Reproduce Figures 4–6

This notebook loads `data/results/accuracy_vs_noise.csv` (produced by `src/evaluation/evaluate.py`)
and regenerates the figures from the paper.

**Requirements:** `pandas`, `matplotlib`, `numpy`

```bash
pip install pandas matplotlib numpy
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

ROOT = Path('..').resolve()
RESULTS = ROOT / 'data' / 'results' / 'accuracy_vs_noise.csv'

df = pd.read_csv(RESULTS)
print(f'Loaded {len(df)} rows')
df.head()

## Figure 4 — Token Reconstruction Accuracy vs. Noise Level

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

models = {
    'gpt2':   ('GPT-2',       'C0', '-',  'o'),
    'bert':   ('BERT-base',   'C2', '-',  '^'),
    'llama':  ('LLaMA-2-sim', 'C3', '-',  'D'),
}

for key, (label, color, ls, marker) in models.items():
    sub = df[df['model'] == key].sort_values('noise_level')
    ax.errorbar(
        sub['noise_level'], sub['top1_mean'],
        yerr=sub['top1_std'], label=f'{label} Top-1',
        color=color, linestyle=ls, marker=marker, capsize=3
    )

# Li et al. (2024) baseline
li_noise = [0, 2, 4, 6, 8, 10]
li_acc   = [48.3, 41.2, 32.8, 25.6, 20.4, 16.9]
li_err   = [3.8,  3.5,  3.9,  4.1,  4.3,  4.4]
ax.errorbar(li_noise, li_acc, yerr=li_err, label='Li et al. (2024) — Timing only',
            color='purple', linestyle='-.', marker='*', capsize=3)

ax.axhline(5.0, color='gray', linestyle=':', label='Random baseline')

ax.set_xlabel('Timing Noise Level σ (ms)')
ax.set_ylabel('Reconstruction Accuracy (%)')
ax.set_ylim(0, 100)
ax.legend(fontsize=8)
ax.set_title('Figure 4: Accuracy vs. Noise Level')
plt.tight_layout()
plt.savefig(ROOT / 'paper' / 'figures' / 'fig4_python.pdf')
plt.show()

## Figure 5 — Security Analysis: Accuracy vs. DP Epsilon

In [ ]:
# Simulated DP sweep data (mirrors the LaTeX pgfplots source)
epsilon = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
trcs_dp = [12.1, 23.4, 38.7, 52.3, 66.1, 71.8]
adv_dp  = [8.3,  18.2, 31.4, 44.6, 59.8, 68.9]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(epsilon, trcs_dp, 'C0-o', label='TRCS (TokenLeak + DP)')
ax.plot(epsilon, adv_dp,  'C3-s', label='Adversarial accuracy + DP')
ax.set_xscale('log')
ax.set_xlabel('DP Privacy Budget ε')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Figure 5: Security Analysis vs. Differential Privacy')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'paper' / 'figures' / 'fig5_python.pdf')
plt.show()

## Figure 6 — Scalability: Accuracy vs. Vocabulary Size

In [ ]:
vocab_k = [1, 5, 10, 20, 50, 100]
gpt2_v  = [89.2, 82.1, 76.3, 68.5, 57.2, 47.1]
bert_v  = [87.4, 79.8, 73.5, 65.1, 54.3, 44.8]
li_v    = [62.1, 55.4, 48.3, 40.2, 31.7, 24.6]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(vocab_k, gpt2_v, 'C0-o',          label='GPT-2 (TokenLeak)')
ax.plot(vocab_k, bert_v, 'C2-^',          label='BERT-base (TokenLeak)')
ax.plot(vocab_k, li_v,   color='purple',
        linestyle='-.', marker='*',        label='Li et al. (2024) — Timing only')
ax.set_xlabel('Vocabulary Size (×10³ tokens)')
ax.set_ylabel('Top-1 Accuracy (%)')
ax.set_title('Figure 6: Scalability vs. Vocabulary Size')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'paper' / 'figures' / 'fig6_python.pdf')
plt.show()